# Эксперимент: VGGish + SVM (baseline)

**Модель:** SVM (RBF) на агрегированных VGGish-эмбеддингах (mean + std)  
**Группа:** 01_baselines

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import mlflow
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils
from shared.evaluate import find_optimal_threshold, evaluate, evaluate_cv_folds, print_cv_summary
from shared.data_utils import build_feature_matrix, get_cv_folds, get_durations
from shared.results_utils import save_result_csv
from shared.mlflow_utils import start_run, log_cv_fold

/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
vggish_model = torch.hub.load("harritaylor/torchvggish", "vggish")
vggish_model.eval()
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
vggish_model = vggish_model.to(device)

def extract_vggish(path: str) -> np.ndarray:
    with torch.no_grad():
        emb = vggish_model.forward(path).cpu().numpy()
    if emb.ndim == 1:
        emb = emb[np.newaxis, :]
    return np.concatenate([emb.mean(axis=0), emb.std(axis=0)]).astype(np.float32)

Using cache found in /Users/dk/.cache/torch/hub/harritaylor_torchvggish_master
/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/resampy/filters.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_holdout_split()
print(f"Train+Val: {len(paths_trainval)}  |  Holdout Test: {len(paths_test)}")

Train+Val: 2359  |  Holdout Test: 417


In [4]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 5.0, 10.0],
    "clf__gamma": ["scale", "auto", 0.01, 0.1],
}

with start_run("exp_vggish_svm", group="01_baselines"):

    mlflow.log_params({
        "model":        "SVM RBF",
        "features":     "VGGish mean+std (256-dim)",
        "cv_folds":     config.CV_N_SPLITS,
        "class_weight": "balanced",
    })

    fold_results = []
    t0 = time.perf_counter()

    for paths_tr, paths_val, labels_tr, labels_val, letters_tr, letters_val, fold in \
            get_cv_folds(paths_trainval, labels_trainval, letters_trainval):

        print(f"\nФолд {fold+1}/{config.CV_N_SPLITS}")
        X_tr  = build_feature_matrix(paths_tr,  extract_vggish)
        X_val = build_feature_matrix(paths_val, extract_vggish)
        X_tr  = np.hstack([X_tr,  letters_tr,  get_durations(paths_tr).reshape(-1, 1)])
        X_val = np.hstack([X_val, letters_val, get_durations(paths_val).reshape(-1, 1)])

        grid = GridSearchCV(pipeline, param_grid, cv=3,
                            scoring="f1_macro", refit=True, n_jobs=-1)
        grid.fit(X_tr, labels_tr)
        clf = grid.best_estimator_

        y_proba_val = clf.predict_proba(X_val)[:, config.CLASS_BAD]
        thr = find_optimal_threshold(labels_val, y_proba_val)
        metrics = evaluate(labels_val, y_proba_val, threshold=thr, verbose=True)
        fold_results.append(metrics)
        log_cv_fold(fold, f1_bad=metrics["f1_bad"], f1_macro=metrics["f1_macro"],
                    roc_auc=metrics["roc_auc"], threshold=thr)

    train_time_sec = time.perf_counter() - t0
    cv_agg = evaluate_cv_folds(fold_results)
    print_cv_summary(cv_agg)

    X_trainval = build_feature_matrix(paths_trainval, extract_vggish)
    X_test     = build_feature_matrix(paths_test,     extract_vggish)
    X_trainval = np.hstack([X_trainval, letters_trainval, get_durations(paths_trainval).reshape(-1, 1)])
    X_test     = np.hstack([X_test,     letters_test,     get_durations(paths_test).reshape(-1, 1)])

    final_grid = GridSearchCV(pipeline, param_grid, cv=5,
                              scoring="f1_macro", refit=True, n_jobs=-1)
    final_grid.fit(X_trainval, labels_trainval)
    mlflow.log_params({f"best_{k}": v for k, v in final_grid.best_params_.items()})

    cv_threshold = cv_agg["threshold_mean"]
    y_proba_test = final_grid.best_estimator_.predict_proba(X_test)[:, config.CLASS_BAD]
    test_metrics = evaluate(labels_test, y_proba_test, threshold=cv_threshold, verbose=True)

    save_result_csv(
        exp_dir=exp_dir,
        experiment_id="exp_vggish_svm",
        experiment_name="VGGish + SVM (baseline)",
        model="VGGish mean+std + SVM RBF",
        accuracy=test_metrics["accuracy"],
        f1_macro=test_metrics["f1_macro"],
        f1_bad=test_metrics["f1_bad"],
        roc_auc=test_metrics["roc_auc"],
        precision_bad=test_metrics["precision_bad"],
        recall_bad=test_metrics["recall_bad"],
        threshold=test_metrics["threshold"],
        cv_f1_bad_std=cv_agg["f1_bad_std"],
        cv_f1_macro_std=cv_agg["f1_macro_std"],
        notes=f"5-fold CV + holdout | best={final_grid.best_params_} | thr={cv_threshold:.2f}",
        train_time_sec=train_time_sec,
    )
    print("Результаты сохранены")

df_folds = pd.DataFrame(fold_results)
df_folds.index = [f"Fold {i+1}" for i in range(len(fold_results))]
display(df_folds[["accuracy", "f1_macro", "f1_bad", "roc_auc", "threshold"]])


Фолд 1/5


/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
/Users/dk/miniconda3/envs/vkr/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Users/dk/miniconda3/envs/vkr/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_reso

              precision    recall  f1-score   support

        good       0.85      0.65      0.74       318
         bad       0.51      0.75      0.61       154

    accuracy                           0.69       472
   macro avg       0.68      0.70      0.67       472
weighted avg       0.74      0.69      0.70       472

Threshold : 0.28
Accuracy  : 0.6864
F1-macro  : 0.6741
F1-bad    : 0.6105
ROC-AUC   : 0.7727

Фолд 2/5


/Users/dk/miniconda3/envs/vkr/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Users/dk/miniconda3/envs/vkr/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Users/dk/miniconda3/envs/vkr/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package

              precision    recall  f1-score   support

        good       0.90      0.46      0.61       319
         bad       0.44      0.89      0.59       153

    accuracy                           0.60       472
   macro avg       0.67      0.68      0.60       472
weighted avg       0.75      0.60      0.61       472

Threshold : 0.19
Accuracy  : 0.6017
F1-macro  : 0.6014
F1-bad    : 0.5913
ROC-AUC   : 0.7490

Фолд 3/5


...

/Users/dk/miniconda3/envs/vkr/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Users/dk/miniconda3/envs/vkr/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Users/dk/miniconda3/envs/vkr/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package

              precision    recall  f1-score   support

        good       0.87      0.74      0.80       318
         bad       0.59      0.77      0.67       153

    accuracy                           0.75       471
   macro avg       0.73      0.76      0.73       471
weighted avg       0.78      0.75      0.76       471

Threshold : 0.33
Accuracy  : 0.7495
F1-macro  : 0.7330
F1-bad    : 0.6667
ROC-AUC   : 0.8276
Метрика                  mean    ± std
----------------------------------------
accuracy               0.6787   0.0645
f1_macro               0.6654   0.0539
f1_bad                 0.6115   0.0306
roc_auc                0.7724   0.0317
precision_bad          0.5195   0.0700
recall_bad             0.7703   0.0911

Средний оптимальный порог: 0.28


/Users/dk/miniconda3/envs/vkr/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Users/dk/miniconda3/envs/vkr/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Users/dk/miniconda3/envs/vkr/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package

              precision    recall  f1-score   support

        good       0.85      0.70      0.77       282
         bad       0.54      0.75      0.63       135

    accuracy                           0.71       417
   macro avg       0.70      0.72      0.70       417
weighted avg       0.75      0.71      0.72       417

Threshold : 0.28
Accuracy  : 0.7146
F1-macro  : 0.6987
F1-bad    : 0.6293
ROC-AUC   : 0.8050
Результаты сохранены


,accuracy,f1_macro,f1_bad,roc_auc,threshold
Fold 1,0.686441,0.674057,0.610526,0.772686,0.28
Fold 2,0.601695,0.601437,0.591304,0.749042,0.19
Fold 3,0.608051,0.605884,0.576659,0.735038,0.22
Fold 4,0.747881,0.712782,0.612378,0.777798,0.36
Fold 5,0.749469,0.732993,0.666667,0.827558,0.33
